# Paper values from the saved grouped run (Tables 2 and 3)

This notebook **does not train, decode, align or call the LLM, and recomputes no metric**. It reads the tables that the full grouped run already saved (run `20261112T053554Z_7b6c322ff5`), checks them against the run's own SHA-256 manifest, and prints each value the camera-ready paper uses, with the file and row it comes from.

**Input:** add the run archive `iciss_fresh_full_20261112T053554Z_7b6c322ff5.zip` as a Kaggle Dataset (*Add Input → Upload*). Kaggle extracts the zip automatically; the notebook also accepts the zip unextracted.

The only operations applied are multiplying proportions by 100 and rounding to two decimals.

In [ ]:
import json, hashlib, zipfile
from pathlib import Path
import pandas as pd

SEARCH_ROOTS = [Path("/kaggle/input"), Path(".")]
RUN_ID = "20261112T053554Z_7b6c322ff5"

def find_run_dir():
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        for hit in root.rglob("asr_accuracy.csv"):
            if hit.parent.name == "tables" and (hit.parent.parent / "run_config.json").exists():
                return hit.parent.parent
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        for z in root.rglob("iciss_fresh_full_*.zip"):
            out = Path("/kaggle/working/run") if Path("/kaggle/working").exists() else Path("run")
            with zipfile.ZipFile(z) as f:
                f.extractall(out)
            return out
    raise FileNotFoundError("Run archive not found. Add iciss_fresh_full_*.zip as a dataset input.")

RUN = find_run_dir()
T = RUN / "tables"
print("Run folder:", RUN)

## 1. Confirm the files are the original run outputs

In [ ]:
cfg = json.loads((RUN / "run_config.json").read_text())
assert cfg["run_id"] == RUN_ID, f"Unexpected run: {cfg['run_id']}"
print("run_id   :", cfg["run_id"])
print("protocol :", cfg["protocol"])
print("model    :", cfg["model_repo"], "@", cfg["model_revision"])
print("LLM      :", cfg["config"]["LLM_MODEL"], "| bootstrap draws:", cfg["n_boot"], "| permutations:", cfg["n_perm"], "| seed:", cfg["seed"])

manifest = json.loads((RUN / "file_hashes.json").read_text())
USED = ["tables/asr_accuracy.csv", "tables/llm_arm_summary.csv", "tables/paired_correction_contrasts.csv",
        "tables/localization.csv", "tables/timing_paired_differences.csv", "tables/timing_marker_agreement.csv",
        "tables/utterance_association.csv", "tables/asr_by_speaker.csv", "threshold_lock.json",
        "training_summary.json", "split_audit.json"]
rows = []
for rel in USED:
    digest = hashlib.sha256((RUN / rel).read_bytes()).hexdigest()
    rows.append({"file": rel, "in_manifest": rel in manifest, "sha256_matches": manifest.get(rel) == digest})
check = pd.DataFrame(rows)
display(check)
assert check.sha256_matches.all(), "A file differs from the run's hash manifest."

## 2. Table 2: ASR on the recording-grouped test set

ASR rows come from `tables/asr_accuracy.csv` with the paper's normalizer (`norm_del`: punctuation deleted). The LLM row comes from `tables/llm_arm_summary.csv` (population `whole`, condition `visible`).

The two files compute MER with different alignment tie-breaks (the run notes this), so MER can differ in the second decimal between them. WER and CER are identical in both.

In [ ]:
pct = lambda x: round(100 * float(x), 2)
acc = pd.read_csv(T / "asr_accuracy.csv")
arm = pd.read_csv(T / "llm_arm_summary.csv")
POP = {"all": "All", "dysarthria": "Dysarthric", "healthy": "Healthy"}

t2 = []
for system, label in [("zero_shot", "Zero-shot Whisper-tiny"), ("asr", "Fine-tuned Whisper-tiny")]:
    for pop in ["all", "dysarthria", "healthy"]:
        r = acc[(acc.system == system) & (acc.population == pop) & (acc.normalizer == "norm_del")].iloc[0]
        t2.append({"System": label, "Speakers": POP[pop], "n": int(r.recordings), "Ref. words": int(r.words),
                   "WER (%)": pct(r.WER), "CER (%)": pct(r.CER), "MER (%)": pct(r.MER),
                   "source": f"asr_accuracy.csv [{system}, {pop}, norm_del]"})
for group in ["all", "dysarthria"]:
    r = arm[(arm.group == group) & (arm.population == "whole") & (arm.arm == "visible")].iloc[0]
    t2.append({"System": "+ pause-guided LLM (0.30 s)", "Speakers": POP[group], "n": int(r.recordings), "Ref. words": int(r.words),
               "WER (%)": pct(r.WER), "CER (%)": pct(r.CER), "MER (%)": pct(r.MER),
               "source": f"llm_arm_summary.csv [{group}, whole, visible]"})
table2 = pd.DataFrame(t2)
display(table2)

## 3. Table 3: paired LLM-correction comparisons (dysarthric test takes)

In [ ]:
pc = pd.read_csv(T / "paired_correction_contrasts.csv")
WANT = [("whole",   "untouched", "visible", "Untouched → visible"),
        ("whole",   "untouched", "hidden",  "Untouched → hidden"),
        ("marked",  "untouched", "visible", "Untouched → visible (marked takes)"),
        ("marked",  "hidden",    "visible", "Hidden → visible (same windows)"),
        ("matched", "random",    "visible", "Random → measured locations"),
        ("matched", "hidden",    "visible", "Hidden → visible (matched takes)")]
t3 = []
for pop, before, after, label in WANT:
    r = pc[(pc.group == "dysarthria") & (pc.population == pop) & (pc.before == before) & (pc.after == after)].iloc[0]
    t3.append({"Comparison": label, "Takes": int(r.n), "Ref. words": int(r.ref_words),
               "Errors": f"{int(r.errors_before)} → {int(r.errors_after)}",
               "Impr./Wors./Tied": f"{int(r.improved)}/{int(r.worsened)}/{int(r.tied)}",
               "WER gain (pts)": pct(r.wer_improvement), "95% CI": f"[{pct(r.lo):.2f}, {pct(r.hi):.2f}]",
               "Sign p": round(float(r.sign_p), 3),
               "source": f"paired_correction_contrasts.csv [dysarthria, {pop}, {before}→{after}]"})
table3 = pd.DataFrame(t3)
display(table3)

print("Arm-level error counts on the 27 matched takes (llm_arm_summary.csv, dysarthria, matched):")
m = arm[(arm.group == "dysarthria") & (arm.population == "matched")][["arm", "recordings", "words", "WER", "edited", "calls"]].copy()
m["WER (%)"] = m.WER.map(pct)
display(m.drop(columns="WER"))

## 4. Values quoted in the text

In [ ]:
loc = pd.read_csv(T / "localization.csv").set_index(["population", "source"])
tpd = pd.read_csv(T / "timing_paired_differences.csv").set_index(["population", "metric"])
agr = pd.read_csv(T / "timing_marker_agreement.csv").set_index("population")
ua  = pd.read_csv(T / "utterance_association.csv")
lock = json.loads((RUN / "threshold_lock.json").read_text())
ts   = json.loads((RUN / "training_summary.json").read_text())
sa   = json.loads((RUN / "split_audit.json").read_text())

V = []
def add(name, value, source): V.append({"value": name, "number": value, "source": source})

add("Grouped split seed", sa["seed"], "split_audit.json:seed")
add("Partitions (train/val/calib/test)", "/".join(str(sa["counts"][k]["recordings"]) for k in ["asr_train", "asr_validation", "pause_calibration", "final_test"]), "split_audit.json:counts")
add("Test dysarthric / control", f'{sa["primary_files_by_speaker_group"]["final_test"]["dysarthria"]} / {sa["primary_files_by_speaker_group"]["final_test"]["healthy"]}', "split_audit.json")
add("Primary channels head / array", f'{sa["primary_channels"]["headMic"]} / {sa["primary_channels"]["arrayMic"]}', "split_audit.json:primary_channels")
add("Excluded by duration", sa["reasons"]["outside_duration_limits"], "split_audit.json:reasons")
add("Test takes repeating a training prompt", f'{sa["final_files_with_exact_prompt_in_training"]} of {sa["final_primary_files"]}', "split_audit.json")
add("Recording overlap between partitions", sum(sa["recording_overlaps"].values()), "split_audit.json:recording_overlaps")
add("Selected step / validation WER (%)", f'{ts["selected_step"]} / {pct(ts["selection_wer"])}', "training_summary.json")
add("Threshold tau (s) / source", f'{lock["tau"]} / {lock["source"]}', "threshold_lock.json")
add("Calibration takes / boundaries / aligned", f'{lock["calibration_n"]} / {lock["boundaries"]} / {lock["status_counts"]["aligned"]}', "threshold_lock.json")

r = loc.loc[("dysarthria", "hyp_native")]
add("Dysarthric boundaries / markers / marked takes", f"{int(r.boundaries)} / {int(r.markers)} / {int(r.recordings_marked)}", "localization.csv [dysarthria, hyp_native]")
add("Near-error markers; reliability vs chance (%)", f"{int(r.near_markers)}; {pct(r.reliability)} vs {pct(r.chance)}", "localization.csv [dysarthria, hyp_native]")
add("Lift (pts) [95% CI]; permutation p", f"{pct(r.lift)} [{pct(r.lift_lo)}, {pct(r.lift_hi)}]; {r.p_reliability:.3f}", "localization.csv [dysarthria, hyp_native]")
add("Errors covered; coverage vs random (%)", f"{int(r.covered_errors)} of {int(r.errors)}; {pct(r.coverage)} vs {pct(r.random_coverage)}", "localization.csv [dysarthria, hyp_native]")
for src in ["hyp_matched", "ref_matched"]:
    r = loc.loc[("dysarthria", src)]
    add(f"Shared boundaries, {src}: markers / near / lift [CI]; p", f"{int(r.boundaries)}: {int(r.markers)} / {int(r.near_markers)} / {pct(r.lift)} [{pct(r.lift_lo)}, {pct(r.lift_hi)}]; {r.p_reliability:.3f}", f"localization.csv [dysarthria, {src}]")
r = tpd.loc[("dysarthria", "lift")]
add("Reference minus hypothesis lift (pts) [CI]", f"{pct(r.reference_minus_hypothesis)} [{pct(r.lo)}, {pct(r.hi)}]", "timing_paired_differences.csv [dysarthria, lift]")
r = agr.loc["dysarthria"]
add("Marked in both / hyp only / ref only", f"{int(r.both)} / {int(r.only_hyp)} / {int(r.only_ref)}", "timing_marker_agreement.csv [dysarthria]")
for flag in [True, False]:
    r = ua[(ua.population == "dysarthria") & (ua.flagged == flag)].iloc[0]
    add(f"Multiword dysarthric takes, marked={flag}: n / any error (%) / excluded", f"{int(r.n)} / {pct(r.p_any_error)} / {int(r.excluded_ctc)}", "utterance_association.csv")
r = arm[(arm.group == "dysarthria") & (arm.population == "marked") & (arm.arm == "untouched")].iloc[0]
add("WER of marked dysarthric takes (%)", pct(r.WER), "llm_arm_summary.csv [dysarthria, marked, untouched]")
r = arm[(arm.group == "dysarthria") & (arm.population == "whole") & (arm.arm == "visible")].iloc[0]
add("Takes edited / calls, visible markers (dysarthric)", f"{int(r.edited)} / {int(r.calls)}", "llm_arm_summary.csv [dysarthria, whole, visible]")
calls = arm[(arm.group == "all") & (arm.population == "whole")].set_index("arm")["calls"]
add("Regional calls visible / hidden / random", f'{int(calls["visible"])} / {int(calls["hidden"])} / {int(calls["random"])}', "llm_arm_summary.csv [all, whole]")

values = pd.DataFrame(V)
pd.set_option("display.max_colwidth", 120)
display(values)

## 5. Guard rejections and per-speaker WER (supporting evidence)

In [ ]:
resp = json.loads((RUN / "llm_region_responses.json").read_text())
items = resp if isinstance(resp, list) else list(resp.values())
flat = []
for it in items:
    if isinstance(it, dict):
        flat.append(it)
    elif isinstance(it, list):
        flat.extend(x for x in it if isinstance(x, dict))
g = pd.DataFrame(flat)
if {"arm", "guard_reason"} <= set(g.columns):
    display(g.groupby(["arm", "guard_reason"]).size().unstack(fill_value=0))
else:
    print("Columns in llm_region_responses.json:", list(g.columns)[:20])

spk = pd.read_csv(T / "asr_by_speaker.csv")
spk["WER (%)"] = spk.WER.map(pct)
display(spk.pivot(index="speaker", columns="system", values="WER (%)"))

## 6. Save everything for the paper

In [ ]:
out = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
table2.to_csv(out / "paper_table2.csv", index=False)
table3.to_csv(out / "paper_table3.csv", index=False)
values.to_csv(out / "paper_text_values.csv", index=False)
print("Saved:", [p.name for p in out.glob("paper_*.csv")])